# <center> Some plots from the DEA results

## <center> Description

- env: squidpy_v06

## <center> Set-up

#### **libs**

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc
import sys
import math
from pathlib import Path
from scipy import sparse

# When this notebook is exported and run as a script, add the project root to sys.path
# so local workflow.* imports behave like they do in the notebook kernel.
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[3]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

## plotting
import plotnine as p9
import seaborn as sns
import matplotlib.pyplot as plt

#### **paths**

In [2]:
from pathlib import Path

if "snakemake" in globals():
    # in_metadata = Path(snakemake.input["in_metadata"])
    # out_metadata = Path(snakemake.output["out_metadata"])
    print('Hello World')
else:
    ## get paths
    ROOT = get_project_root(globals())
    idx=0 # 0-1
    runName_lst=['HR_case__IFNGpos_vs_IFNGneg', 'HR_control__IFNGpos_vs_IFNGneg']
    run_name=runName_lst[idx]
    
    ## input
    in_adata = ROOT / "output/preprocessing/objects/QC/adata.h5ad"
    in_deg = ROOT / f"output/DEA/objects/pyDESeq2_byGrp/{run_name}/deg.tsv"
    
    ## output
    ### plots
    out_plots = ROOT / f"output/DEA/plots/plot_DEA_byGrp/{run_name}"
    out_plots.mkdir(parents=True, exist_ok=True)


## <center> Main

#### **Load data**

In [3]:
# adata

adata = sc.read_h5ad(in_adata)
## remove group B
adata = adata[adata.obs['QC_group']=='A'].copy()
## keep samples in subset (included in the DEA)
if 'HR_case' in run_name:
    adata = adata[adata.obs['disease_group']=='HR_case']
else:
    adata = adata[adata.obs['disease_group']=='HR_control']
## make sure counts are norm
adata.X = adata.layers['cts.deseq2_norm'].copy()
## col with ENSEMBL ID
adata.var['ENS']=adata.var.index
## rename var_names
adata.var_names = [symb if (not pd.isna(symb)) else ENS for ENS,symb in zip(adata.var.index, adata.var['genesymbol'])]
## because some genesymbol won't be unique
adata.var_names_make_unique()
## conv dictionary to re-name degs
conv_dict = {ENS:new_symb for ENS,new_symb in zip(adata.var['ENS'],adata.var.index)}

adata

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/anndata/_core/anndata.py:602: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
/tmp/ipykernel_2309477/765130079.py:12: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
/tmp/ipykernel_2309477/765130079.py:14: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.


AnnData object with n_obs × n_vars = 24 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ENS'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [4]:
# DEA results

deg = pd.read_csv(in_deg,sep='\t',index_col=0)
## rename genesymbol as in adata.var
deg['genesymbol'] = deg.index.map(conv_dict)

deg.head()

,genesymbol,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,contrast,significant_pvalue_0.05,significant_padj_0.05
gene_id,,,,,,,,,,
ENSG00000182326,C1S,1575.330976,2.603546,0.097059,26.824250,1.685002e-158,3.229474e-154,HR_case__IFNGpos_vs_IFNGneg,True,True
ENSG00000128335,APOL2,1829.840173,2.619529,0.102575,25.537680,7.525079e-144,7.211283e-140,HR_case__IFNGpos_vs_IFNGneg,True,True
ENSG00000131203,IDO1,2068.648886,8.282069,0.331990,24.946758,2.315582e-137,1.479348e-133,HR_case__IFNGpos_vs_IFNGneg,True,True
ENSG00000138755,CXCL9,2888.259868,8.373022,0.382825,21.871655,4.837099e-106,2.317696e-102,HR_case__IFNGpos_vs_IFNGneg,True,True
ENSG00000169245,CXCL10,3084.527125,7.455920,0.359402,20.745359,1.350126e-95,5.175305e-92,HR_case__IFNGpos_vs_IFNGneg,True,True


## <center> Plots

#### **Volcano plot**

In [5]:
# QC plot: volcano plot with decoupler

volcano_tab = deg.dropna(subset=["log2FoldChange", "padj"]).copy()
volcano_tab = volcano_tab.set_index("genesymbol", drop=False)

fig = dc.pl.volcano(
    data=volcano_tab,
    x="log2FoldChange",
    y="padj",
    top=20,
    thr_stat=0.5,
    thr_sign=0.05,
    figsize=(7, 6),
    dpi=120,
    return_fig=True,
)
fig.axes[0].set_title(run_name)
fig.savefig(out_plots / "volcano.png", bbox_inches="tight")
fig.savefig(out_plots / "volcano.pdf", bbox_inches="tight")

plt.close(fig)


#### **Plot top DEGs**

In [6]:

topn=20

tmp=adata.copy()
## get topn genes in comparison
keys=deg.sort_values('padj',ascending=True)
keys=keys[keys['padj']<0.05]
keys=keys.head(topn)['genesymbol'].tolist()

As boxplots

In [7]:
## get table with norm counts and metadata in long format
ptab = pd.DataFrame(tmp.layers['cts.deseq2_norm'], index=tmp.obs.index,columns=tmp.var.index)
ptab = (
    ptab[keys].reset_index().
    melt(id_vars=['sample_id'],var_name='genesymbol',value_name='expr'))
### add grp from adata
ptab['grp']=ptab['sample_id'].map( tmp.obs['grp'].to_dict() )
col_dict={
    'HR_case__IFNG_neg':'HR case, IFNG-', 'HR_case__IFNG_pos':'HR case, IFNG+', 
    'HR_control__IFNG_neg':'HR control, IFNG-','HR_control__IFNG_pos':'HR control, IFNG+'}
ptab['grp']=ptab['grp'].map(col_dict)
cat_order=[i for _,i in col_dict.items()]
cat_order=[c for c in cat_order if c in ptab['grp'].unique()]
ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(cat_order,ordered=True)
## add stat and padj
cols=['stat','padj']
for col in cols:
    col_dict={x:y for x,y in zip(deg['genesymbol'],deg[col])}
    ptab[col]=ptab['genesymbol'].map(col_dict)
### plot title (ordered by significance)
ptab['padj']=['{:.2e}'.format(x) for x in ptab['padj']]
ptab['p_title']=[f"{a}\nStat: {round(float(b),2)}\nPadj: {c}" for a,b,c in zip(ptab['genesymbol'],ptab['stat'],ptab['padj'])]
ptab['p_title']=ptab['p_title'].astype('category').cat.reorder_categories(ptab['p_title'].unique(), ordered=True)

ptab.head()

,sample_id,genesymbol,expr,grp,stat,padj,p_title
0,SD1,C1S,249.836544,"HR case, IFNG-",26.82425,3.23e-154,C1S\nStat: 26.82\nPadj: 3.23e-154
1,SD3,C1S,114.422843,"HR case, IFNG-",26.82425,3.23e-154,C1S\nStat: 26.82\nPadj: 3.23e-154
2,SD5,C1S,212.553494,"HR case, IFNG-",26.82425,3.23e-154,C1S\nStat: 26.82\nPadj: 3.23e-154
3,SD7,C1S,66.030012,"HR case, IFNG-",26.82425,3.23e-154,C1S\nStat: 26.82\nPadj: 3.23e-154
4,SD9,C1S,611.087996,"HR case, IFNG-",26.82425,3.23e-154,C1S\nStat: 26.82\nPadj: 3.23e-154


In [8]:
# as boxplot


## palette
pal_name='Oranges' if 'HR_case' in run_name else 'Blues'
pal = {grp:color for grp,color in zip(ptab['grp'].cat.categories,sns.color_palette(pal_name,n_colors=2).as_hex())}

plots_per_col=8
ncols=math.ceil(len(keys)/plots_per_col)
w=2.3*plots_per_col if plots_per_col>=4 else 6
h=4*ncols

contrast='grp'
p=(
    p9.ggplot(ptab, p9.aes(x=contrast, y='expr', group=contrast)) +
    p9.geom_boxplot(outlier_alpha=0,size=0.8,width=0.3,show_legend=False) +
    p9.geom_jitter(p9.aes(fill=contrast), color='black', size=3, width=0.1,alpha=1.0,show_legend=True) + 
    p9.scale_fill_manual(pal) +
    p9.guides(fill=p9.guide_legend(override_aes={'size':6})) +
    p9.theme_classic() +
    p9.labs(title="Expression of selected genes") +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=14), 
        strip_text=p9.element_text(size=12), 
        axis_title=p9.element_text(size=14), 
        axis_text_x=p9.element_blank(), 
        legend_text=p9.element_text(size=12),
        figure_size=(w, h),
    ) + 
    p9.facet_wrap('p_title', scales = 'free', ncol=plots_per_col) 
)
p.save(out_plots / "DEA_boxplot_01.png", width=w, height=h, limitsize=False)
p.save(out_plots / "DEA_boxplot_01.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 18.4 x 12 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/plot_DEA_byGrp/HR_case__IFNGpos_vs_IFNGneg/DEA_boxplot_01.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 18.4 x 12 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/plot_DEA_byGrp/HR_case__IFNGpos_vs_IFNGneg/DEA_boxplot_01.pdf
